### Load packages and data

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
from statsmodels.stats.multitest import multipletests
import pickle

In [ ]:
# statistical annotation
def convert_pvalue_to_asterisks(pvalue):
    if pvalue <= 0.0001:
        return " **** "
    elif pvalue <= 0.001:
        return " *** "
    elif pvalue <= 0.01:
        return "  **  "
    elif pvalue <= 0.05:
        return "   *  "
    return "  ns  "

In [ ]:
# load all behavioral data: 1D task and 2D task
raw_path = '../../data/behavior'
subj_list = ['A03','A04','A05','A07','A08','A10','A12','A13','A14','A15','A16','A17']
    

In [ ]:
cmap_pe = ['#ff9eaa','#3aa6b9','#808080','#6aa84f']  # SPE, RPE, nonPE, dualPE

In [ ]:
cm = 1/2.54

In [ ]:
with open(f'{raw_path}/beh_binary.pkl', "rb") as f:  # Python 3: open(..., 'wb')
    c_df, all_evals_df = pickle.load(f)

with open(f'{raw_path}/beh_navigation.pkl', "rb") as f:  # Python 3: open(..., 'wb')
    t2_srpe_df = pickle.load(f)

# Binary choice task

## Consistency ratio

In [ ]:
# signifiance above 0.9
ratio_s = c_df[c_df['c_type']=='state']['ratio']
ts,ps = scipy.stats.ttest_1samp(ratio_s, 0.9, alternative='two-sided')
ratio_r = c_df[c_df['c_type']=='reward']['ratio']
tr,pr = scipy.stats.ttest_1samp(ratio_r, 0.9, alternative='two-sided')

multipletests([ps, pr], method='fdr_bh')

In [ ]:
fig,(ax1,ax2) = plt.subplots(nrows=2, ncols=1, figsize=(4*cm, 5*cm)) # top: ax1, bottom: ax2
fig.subplots_adjust(hspace=0.1)  # adjust space between Axes

sns.pointplot(ax=ax1, data=c_df, x='c_type', hue='c_type', order=['reward','state'], y='ratio', palette=[cmap_pe[0],cmap_pe[1]], \
                   capsize=.2, err_kws={'linewidth': 1}, scale=.6, legend=False)

ax1.axhline(y=0.9, color=".7", dashes=(2, 1), zorder=0)
ax2.set_ylim([.48,.62])
ax1.set_ylim([.85,1])
ax1.set_xlim([-.7,1.7])
ax2.set_xlim([-.7,1.7])
ax1.set_xticks(ax1.get_xticks(), ax1.get_xticklabels(), fontsize=8)
ax1.set_yticks([0.9, 1.0], [0.9, 1.0], fontsize=8)
ax2.set_xticks([0, 1], ['state', 'reward'], fontsize=8)
ax2.set_yticks([0.5], [0.5], fontsize=8)


# hide the spines between ax and ax2
ax1.spines.bottom.set_visible(False)
ax1.axes.get_xaxis().set_visible(False)
ax2.spines.top.set_visible(False)
ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False)  # don't put tick labels at the top
ax2.xaxis.tick_bottom()

# Now, let's turn towards the cut-out slanted lines.
d = .3  # proportion of vertical to horizontal extent of the slanted line
kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
              linestyle="none", color='k', mec='k', mew=1, clip_on=False)
ax1.plot([0, 1], [0, 0], transform=ax1.transAxes, **kwargs)
ax2.plot([0, 1], [1, 1], transform=ax2.transAxes, **kwargs)

plt.tight_layout()
plt.gcf().set_dpi(300)
plt.savefig('../../results/fig3a.png')
plt.show()

## Correct submit ratio

In [ ]:
# check !for each session!: ratio of the correct evaluation
cm = 1/2.54
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4*cm, 4*cm))
ax = sns.lineplot(data=all_evals_df, x='session', y='attention', color='black',\
                   errorbar='se',err_style='bars', linewidth=1)

ax.set_yticks([0.45, 0.5,0.6,0.7,0.8,0.9,1.0],['', 0.5,0.6,0.7,0.8,0.9,1.0],fontsize=8)
ax.set_xticks([1,5,10,15],[1,5,10,15],fontsize=8)
# ax.set_xticklabels(range(1,m+2),fontsize=18)
ax.axhline(y=0.5, color=".7", dashes=(2, 1), zorder=0,label='Chance Level=0.5')
ax.set_xlabel('Session', fontsize=8)
ax.set_ylabel('Correct submit ratio', fontsize=8)

plt.tight_layout()
plt.gcf().set_dpi(300)
plt.savefig('../../results/fig3b.png')
plt.show()

# Navigation task

## Response time

In [ ]:
t2_response_time_by_subj_df = t2_srpe_df.groupby(['subj', 'block'], as_index=False)['resp_time'].mean()
# t2_response_time_by_subj_df

In [ ]:
cm = 1/2.54
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4*cm, 4*cm))
this_df = t2_response_time_by_subj_df

ax = sns.lineplot(data=this_df, x='block', y='resp_time', color='black',\
                   errorbar='se',err_style='bars', linewidth=1)

ax.set_xlabel('Session', fontsize=8)
ax.set_xticks([1,5,10,15],[1,5,10,15], fontsize=8)
ax.set_ylabel('Response time [s]', fontsize=8)
ax.set_yticks(ax.get_yticks(), ax.get_yticklabels(), fontsize=8)
ax.set_ylim([0.9,2.6])

plt.tight_layout()
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig3c.svg', transparent=True, format='svg')

## Prediction errors

In [ ]:
df = t2_srpe_df
df['trial'] = df['trial'].apply(str)
# Example of converting 'subj' to a string if it contains arrays or lists
df['SPE'] = df['SPE'].astype(int)
df['RPE'] = df['RPE'].astype(int)

t2_by_subj_df = df.groupby(['subj', 'block', 'trial'], as_index=False)[['SPE', 'RPE']].sum()
t2_mean_by_subj_df = t2_by_subj_df.groupby(['subj', 'trial'])[['SPE', 'RPE']].mean().reset_index()

In [ ]:
import pingouin as pg
df = t2_mean_by_subj_df
df['trial'] = df['trial'].astype(float).astype(int)
stats_s = pg.rm_corr(data=df, x='trial', y='SPE', subject='subj')
print(stats_s)
# g = pg.plot_rm_corr(data=df, x='trial', y='SPE', subject='subj')

stats_r = pg.rm_corr(data=df, x='trial', y='RPE', subject='subj')
print(stats_r)
# g = pg.plot_rm_corr(data=df, x='trial', y='RPE', subject='subj')

_, p_corrected,_,_ = multipletests([stats_s.pval[0],stats_r.pval[0]], method='fdr_bh')
p_corrected

In [ ]:
cm = 1/2.54
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.5*cm, 4*cm))
this_df = t2_mean_by_subj_df

ax = sns.lineplot(data=this_df, x='trial', y='RPE', color=cmap_pe[1], alpha=0.7,\
                   errorbar='se',err_style='bars', linewidth=1)
sns.lineplot(ax=ax, data=this_df, x='trial', y='SPE', color=cmap_pe[0],  alpha=0.7,\
                   errorbar='se',err_style='bars', linewidth=1)

ax.set_ylim([2.0,4.0])

ax.set_xlabel('Trial', fontsize=8)
ax.set_ylabel('Prediction errors', fontsize=8)
ax.set_yticks(ax.get_yticks(), ax.get_yticklabels(), fontsize=8)
ax.set_xticks([1,2,3,4,5],[1,2,3,4,5], fontsize=8)
ax.legend(['Goal', 'Action'], bbox_to_anchor=(0.3, 1), fontsize=8)

plt.tight_layout()
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig3d.svg', transparent=True, format='svg')